# 01 — Setup và chuẩn bị dataset
Mặc định `QUICK_TEST=True`: tạo 3 mesh synthetic, không tải ShapeNetCore. Trên Colab hãy chọn GPU, thêm secret `HF_TOKEN`, rồi bấm **Run all**. Để chạy Chair thật, dùng `configs/chair_colab.yaml` sau khi được cấp quyền. Chạy lại notebook là an toàn: download/extraction/preparation hỗ trợ resume.

In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path
STARTED = time.perf_counter()
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, cwd / 'image-to-3d-shapenet', *cwd.parents] if (p / 'configs' / 'quick_test.yaml').is_file()), None)
if PROJECT_ROOT is None:
    clone_target = Path('/content/image-to-3d-shapenet')
    try:
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/dinhcongtiendat2003/image-to-3d-shapenet.git', str(clone_target)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Không tìm thấy hoặc clone được repository từ GitHub.') from exc
    PROJECT_ROOT = clone_target
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError('Cài dependency thất bại. Hãy Restart runtime và Run all lại.') from exc
import torch
CUDA = torch.cuda.is_available()
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3 if CUDA else 0.0
HF_TOKEN_AVAILABLE = bool(os.getenv('HF_TOKEN'))
try:
    from google.colab import userdata
    HF_TOKEN_AVAILABLE = HF_TOKEN_AVAILABLE or bool(userdata.get('HF_TOKEN'))
except Exception:
    pass
DRIVE_ROOT = None
try:
    from google.colab import drive
    drive.mount(os.getenv('GOOGLE_DRIVE_MOUNT', '/content/drive'))
    DRIVE_ROOT = Path(os.getenv('GOOGLE_DRIVE_MOUNT', '/content/drive')) / 'MyDrive' / 'image-to-3d-shapenet'
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
except Exception as exc:
    DRIVE_ROOT = None
    print(f'Google Drive mount không khả dụng ({type(exc).__name__}); tiếp tục lưu local trong /content.')
print({'project': str(PROJECT_ROOT), 'cuda': CUDA, 'gpu': torch.cuda.get_device_name(0) if CUDA else None, 'vram_gb': round(VRAM_GB, 2), 'hf_token_available': HF_TOKEN_AVAILABLE, 'drive': str(DRIVE_ROOT) if DRIVE_ROOT else None})


In [ ]:
QUICK_TEST = True
CONFIG = PROJECT_ROOT / 'configs' / ('quick_test.yaml' if QUICK_TEST else 'chair_colab.yaml')
if not QUICK_TEST and not HF_TOKEN_AVAILABLE:
    raise RuntimeError('ShapeNetCore thật cần quyền gated và secret HF_TOKEN loại Read.')
command = [sys.executable, str(PROJECT_ROOT / 'scripts' / 'prepare_dataset.py'), '--config', str(CONFIG)]
try:
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)  # xác minh resume
except subprocess.CalledProcessError as exc:
    raise RuntimeError('Chuẩn bị dataset thất bại. Kiểm tra HF_TOKEN/quyền ShapeNetCore, RAM và dung lượng đĩa.') from exc


In [ ]:
import json
report_path = PROJECT_ROOT / 'outputs' / 'quick_test' / 'dataset_preparation_report.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
disk = shutil.disk_usage(PROJECT_ROOT)
peak_vram = torch.cuda.max_memory_allocated() / 1024**3 if CUDA else 0.0
print(report)
print({'elapsed_seconds': round(time.perf_counter() - STARTED, 2), 'peak_vram_gb': round(peak_vram, 3), 'disk_free_gb': round(disk.free / 1024**3, 2)})
if DRIVE_ROOT:
    target = DRIVE_ROOT / 'audit' / report_path.name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(report_path, target)
